# 02 — Quantization + TensorRT engine build

Take the verified ONNX graph from `01_export` and (a) build an FP16 TensorRT engine from it, and (b) produce a quantized model (AWQ / GPTQ / INT8).

**GPU required** for every work cell below. Validate with the tiny dev model on a free T4, then switch to the real 8B config on an A100. FP8 needs an H100.

In [ ]:
# ── Environment check ──────────────────────────────────────────
import sys
sys.path.insert(0, "/content/llm-inference-optimizer")  # Colab path
sys.path.insert(0, "..")                                  # local path fallback

from src.utils.env import get_env_info
info = get_env_info()
print(f"Device   : {info['device']}")
print(f"GPU      : {info['gpu_name']}")
print(f"CUDA     : {info['cuda_version']}")
print(f"PyTorch  : {info['torch_version']}")
print(f"Colab    : {info['is_colab']}")

## 1. Choose the model

`USE_TINY` exports/quantizes an ungated tiny LLaMA to validate the pipeline (no HF token). Set it to `False` for the real LLaMA 3 8B (needs `HF_TOKEN` + A100).

In [ ]:
from src.utils.config import load_model_config

USE_TINY = True  # flip to False for the real 8B

if USE_TINY:
    MODEL = "hf-internal-testing/tiny-random-LlamaForCausalLM"
    DTYPE = "fp32"
    TAG = "tiny"
else:
    cfg = load_model_config("llama3_8b")
    MODEL = cfg["model_id"]
    DTYPE = cfg["dtype"]
    TAG = "llama3_8b"
    # Gated model: authenticate first.
    # import os; from huggingface_hub import login; login(os.environ["HF_TOKEN"])
print(f"Model: {MODEL}  dtype: {DTYPE}")

## 2. Export to ONNX (or reuse `01_export` output)

The TensorRT build consumes `model.onnx`. Re-run the export here so the notebook is self-contained; skip if you already have it from `01_export`.

In [ ]:
from src.export.onnx_exporter import export_to_onnx

ONNX_DIR = f"results/onnx/{TAG}"
onnx_path = export_to_onnx(
    model_name_or_path=MODEL,
    output_path=ONNX_DIR,
    dtype=DTYPE,
    verify_export=False,  # parity already covered in 01_export
)
print("ONNX:", onnx_path)

## 3. Build a TensorRT engine

FP16 is implemented. The builder reads the static KV dims straight from the ONNX graph and constructs one optimization profile over batch / sequence / past-length. `int8`/`fp8` are not implemented yet and will raise.

In [ ]:
from src.optimization.trt_builder import build_trt_engine

# On TRT >= 11 the engine precision follows the ONNX dtype (strongly-typed
# networks), so pass the same DTYPE used for the export.
engine_path = build_trt_engine(
    onnx_path=onnx_path,
    output_path=f"results/engines/{TAG}_{DTYPE}.engine",
    precision=DTYPE,
    max_batch_size=8 if USE_TINY else 32,
    max_seq_len=128 if USE_TINY else 2048,
    workspace_gb=2 if USE_TINY else 8,
)
print("Engine:", engine_path)

## 4. Quantize the model

**`int8`** (bitsandbytes) runs in this same environment.

**`awq` / `gptq`** require the SEPARATE quant environment
(`requirements/gpu-quant.txt`, transformers 5.x) — run them in a fresh
Colab session, not this one (their backends conflict with the export/TRT
stack). `fp8` requires an H100 and will raise on T4/A100.

In [ ]:
from src.optimization.quantization import quantize_model

METHOD = "awq"  # "awq" | "gptq" | "int8" | "fp8"
quant_path = quantize_model(
    model_name_or_path=MODEL,
    method=METHOD,
    output_path=f"results/quantized/{TAG}_{METHOD}",
)
print("Quantized:", quant_path)

In [ ]:
# ── Save results to GitHub ─────────────────────────────────────
import subprocess
NOTEBOOK_NAME = "02_optimize"
subprocess.run(["git", "add", "results/", "notebooks/"], check=True)
subprocess.run(["git", "commit", "-m", f"results: {NOTEBOOK_NAME} run"], check=True)
subprocess.run(["git", "push"], check=True)
print("Results pushed to GitHub.")